# RawFileReader integration test

This notebook exercises `RawFileAdapter` against the repository's real `sample.raw` and the bundled .NET 8 RawFileReader assemblies. Run all cells from either the repository root or the `tests` directory.

## Prerequisites

- Python dependencies installed with `pip install -r requirements.txt`
- A .NET 8 runtime available to `pythonnet`
- `sample.raw` present at the repository root


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd().resolve()
repo_root = (
    working_directory.parent
    if working_directory.name == "tests"
    else working_directory
)
if not (repo_root / "rawfilereader").is_dir():
    raise RuntimeError("Run this notebook from the repository root or tests directory")

sys.path.insert(0, str(repo_root))

from rawfilereader import RawFileAdapter


In [ ]:
sample_raw = repo_root / "sample.raw"
assemblies = repo_root / "libs" / "Net8" / "Assemblies"
required_assemblies = (
    "OpenMcdf.dll",
    "OpenMcdf.Extensions.dll",
    "ThermoFisher.CommonCore.Data.dll",
    "ThermoFisher.CommonCore.RawFileReader.dll",
    "ThermoFisher.CommonCore.BackgroundSubtraction.dll",
)

assert sample_raw.is_file(), f"Missing integration fixture: {sample_raw}"
assert sample_raw.stat().st_size > 0, "sample.raw is empty"
for assembly_name in required_assemblies:
    assembly_path = assemblies / assembly_name
    assert assembly_path.is_file(), f"Missing assembly: {assembly_path}"

print(f"RAW fixture: {sample_raw}")
print(f"Assemblies: {assemblies}")


In [ ]:
adapter = RawFileAdapter(str(sample_raw), libs_dir=str(assemblies))
assert not adapter.is_open

with adapter:
    assert adapter.is_open
    first_scan, last_scan = adapter.get_scan_range()
    assert first_scan >= 1
    assert last_scan >= first_scan

    file_info = adapter.get_file_info()
    first_scan_data = adapter.get_centroid_stream(first_scan)

    print(f"File: {file_info.file_name}")
    print(f"Scan range: {first_scan}–{last_scan}")
    print(f"First scan centroid peaks: {len(first_scan_data.masses)}")

assert not adapter.is_open
print("Integration test passed; the native RAW file was closed.")
